# Assignment 02 - Transforms

This assignment covers robustness testing using data transforms with PyTorch and TorchVision.

## Introduction

As we have seen in class, PyTorch and its related packaged provide a powerful means of transforming data in a way that is decoupled from the actual code in our training loop.

The `torch.utils.data.Dataset` and `torch.utils.data.DataLoader` classes provide an interface for us to quickly download standard ML datasets and to make our our datasets which can benefit from the full functionality of PyTorch and its related packages.

When combined with `torch.transforms`, these tools provide a means to transform our data, and to change which transforms we apply without having to make changes to our core training code. This is incredibly useful for data augmentation, domain generalization, and robustness testing.

For data augmentation, transforms can be applied before training to help us increase the diversity in our dataset, and instill expert knowledge into our models by having experts select which transforms we apply to the training data (e.g., applying rotations to a model which must be perspective invariant).

For domain generalization, transforms can be applied during training to help us improve our model's ability to generalize to new domains (environments in which the model must perform its task). For example, varying the color of the background randomly ensures the model is not incorrectly sensitive to background color and will not break in a domain with different background colors.

For robustness testing, transforms can be applied after training to assess our model's robustness to challenging factors in our data, e.g., blur, color changes, perspective changes, and other variations. We can apply transforms to various degrees using the parameters that PyTorch exposes to adjust the transform. This enables us to extract performance curves for our models which go beyond simply displaying a confusion matrix as the output of model training.

This assignment will focus on applying transforms for robustness testing.

## Directions

In this assignment, we will train a model and assess its robustness using PyTorch. To do this, we will carry out the following steps.

1. **Load Data**: This step will require us to import a torch Dataset and use a torch DataLoader to access it. You may use any dataset you like, as long as it is imagery (since this assignment is focused on image transforms). Datasets built into PyTorch may be used. **0.25 points extra credit will be awarded for implementing a custom Dataset.** Be sure to pick a dataset you have the capacity to train a model on in your development environment. To keep the assignment fair grading will not be based on the size of the dataset used in any way. Plot a few of the images to ensure you have loaded the data correctly.

2. **Define the Training Pipeline**: This step will require us to define the class which specifies our neural network architecture, and to define the functions which allow us to train it. We have seen examples of this before, both in class and prior assignments. Be sure to chose an architecture with sufficient capacity to learn the dataset you have chosen in step #1.

3. **Train the Model**: In this step we will use our training functions from step #2 to train a model.

4. **Use Transforms to Robustness Test the Model Against Blur**: In this step, we will robustness test the model against an image blur transform. This step will require us to run model inference on the **test** set repeatedly, increasing the amount of image blur with each run. For each image blur level, we will need to save off the accuracy, then plot the accuracy against the degree of blur (choose and appropriate metric for the degree of blur). The end result of this step should be a plot of accuracy with respect to degree of blur. Be sure to test the model **to failure** to fully characterize its limitations.

5. **Robustness Test the Model Against a Transform of Your Choice**: In this step, we will robustness test the model against a transform of our choice (perhaps perspective changes, color changes, or contrast changes). Like step #4, this step will require us to run model inference on the **test** set repeatedly

## Grading

This assignment is entirely open ended and will not be auto-graded. Each step is worth 20% of the assignment grade. For each step, ensure the following conditions are met to get full credit.

| **Step** | **Criteria for Full Credit** |
|----|----|
| 1. Data Loading |  The data is successfully loaded and a few of the images are visualized.  |
| 2. Training Pipeline Definition | The training pipeline and neural network architecture are defined and error free.  |
| 3. Training | The model training runs for a sufficient number of epochs and the error curve is negative, indicating some learning is taking place. |
| 4. Robustness Test - Blur  | A curve of accuracy with respect to degree of blur is produced and is of professional quality. |
| 5. Robustness Test - Other  | A curve of accuracy with respect to degree of transformation is produced and is of professional quality. |

## Tips

This assignment is intentionally open ended. **No starter code is provided.** The goal of this assignment is to test your ability to put the concepts we have discussed in class into practice.

The steps should be similar, but not necessarily exactly the same, as steps we have discussed in class.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

# Your imports here

import random, numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import GaussianBlur
from torchvision.transforms import ColorJitter



# Reproducibility
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Step 1 - Load Data

Use the cell below (or make additional cells) to load a torch Dataset and use a torch DataLoader to access it. You may use any dataset you like, as long as it is imagery (since this assignment is focused on image transforms). Datasets built into PyTorch may be used.

**0.25 points extra credit will be awarded for implementing a custom Dataset.** Be sure to pick a dataset you have the capacity to train a model on in your development environment. To keep the assignment fair grading will not be based on the size of the dataset used in any way. Plot a few of the images to ensure you have loaded the data correctly.


In [ ]:
# Your code here
MNIST_MEAN, MNIST_STD = (0.1307,), (0.3081,)

# Training transform: ToTensor() scales to [0,1] and adds channel dim; Normalize() standardizes inputs
train_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# Test transform: only ToTensor() and Normalize()
test_base_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# Load datasets and create dataloaders
data_root = "./data"
train_ds = datasets.MNIST(root=data_root, train=True,  download=True, transform=train_tf)
test_ds  = datasets.MNIST(root=data_root, train=False, download=True, transform=test_base_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# visualize 8 samples (unnormalize for display)
inv = transforms.Normalize(mean=[-MNIST_MEAN[0]/MNIST_STD[0]], std=[1/MNIST_STD[0]])
imgs, labels = next(iter(train_loader))
imgs_disp = inv(imgs[:8]).clamp(0,1)

plt.figure(figsize=(10,2))
for i in range(8):
    plt.subplot(1,8,i+1)
    plt.imshow(imgs_disp[i,0].numpy(), cmap="gray")
    plt.title(int(labels[i]))
    plt.axis("off")
plt.tight_layout(); plt.show()


## Step 2 - Define Training Pipeline

Define a training pipeline like those we have seen in class. Define the class which specifies our neural network architecture, and to define the functions which allow us to train it. We have seen examples of this before, both in class and prior assignments.

Be sure to chose an architecture with sufficient capacity to learn the dataset you have chosen in step #1.

Tip: if you see your architecture is insufficient to learn the dataset you have chosen when you run step 3, you may need to return to step #2 to make your architecture wider or deeper.

In [ ]:
# Your code here
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)   # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 28x28 -> 28x28
        self.pool  = nn.MaxPool2d(2,2)                # 28->14, 14->7
        self.drop  = nn.Dropout(0.25)
        self.fc1   = nn.Linear(32*7*7, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, image_batch):
        x = F.relu(self.conv1(image_batch))       # [B,16,28,28]
        x = self.pool(F.relu(self.conv2(x)))      # [B,32,14,14]
        x = self.pool(x)                          # [B,32,7,7]
        x = x.flatten(1)                          # [B,1568]
        x = self.drop(F.relu(self.fc1(x)))        # [B,64]
        return F.log_softmax(self.fc2(x), dim=1)  # [B,10]

# Evaluation helper (no grad, eval mode)
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logp = model(xb)
        loss_sum += F.nll_loss(logp, yb, reduction="sum").item()
        pred = logp.argmax(1)
        total += yb.size(0)
        correct += (pred == yb).sum().item()
    return loss_sum/total, 100.0*correct/total

# Training loop
def train(model, train_loader, test_loader, epochs=5, lr=1e-3, device="cpu"):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    train_losses, test_losses, test_accs = [], [], []
    for ep in range(1, epochs+1):
        model.train(); running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logp = model(xb)
            loss = F.nll_loss(logp, yb)
            loss.backward(); opt.step()
            running += loss.item()
        tl = running/len(train_loader)
        vl, va = evaluate(model, test_loader, device)
        train_losses.append(tl); test_losses.append(vl); test_accs.append(va)
        print(f"Epoch {ep}/{epochs} | train {tl:.4f} | val {vl:.4f} | acc {va:.2f}%")

    # quick curves
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax1.plot(train_losses, label="train loss")
    ax1.plot(test_losses, label="val loss"); ax1.set_xlabel("epoch"); ax1.set_ylabel("NLL loss")
    ax2 = ax1.twinx(); ax2.plot(test_accs, label="val acc (%)", linestyle="--")
    ax2.set_ylabel("accuracy (%)")
    ax1.legend(loc="upper left"); plt.title("Training curves (MNIST CNN)"); plt.show()
    return model


## Step 3 - Train a Model

Kick off your training run here. Be sure to print or plot the error wrt. number of epochs so you can see if your model is learning.


In [ ]:
# Your code here
model = MnistCNN()
model = train(model, train_loader, test_loader, epochs=5, lr=1e-3, device=device)
# Final evaluation
test_loss, test_acc = evaluate(model, test_loader, device)
print(f"Final Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.2f}%")
print("Using:", device)



## Step 4 - Use Transforms to Robustness Test the Model Against Image Blur

Use the cell below (or add more cells), to write code which repeatedly (e.g., in a for-loop) infers your model on your test set. Each time you do so, apply a blur transform with an increasing amount of blur. Compute and save the average accuracy across the entire test set and then plot the accuracy against the degree of blur (choose and appropriate metric for the degree of blur). The end result of this step should be a plot of accuracy with respect to degree of blur. Be sure to test the model **to failure** to fully characterize its limitations.


In [ ]:
# Your code here

def make_test_loader_with(extra_tf):
    # Build a fresh dataset that applies extra_tf BEFORE ToTensor/Normalize
    t = transforms.Compose([
        extra_tf,
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD),
    ])
    ds = datasets.MNIST(root=data_root, train=False, download=False, transform=t)
    return DataLoader(ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

sigmas = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]  # extend until failure
acc_vs_sigma = []

for s in sigmas:
    if s == 0.0:
        extra = transforms.Lambda(lambda img: img)  # identity
    else:
        k = int(max(3, 2*round(3*s)+1))  # odd kernel, ~6*s window
        extra = GaussianBlur(kernel_size=k, sigma=s)
    loader_blur = make_test_loader_with(extra)
    try:
        loss, acc = evaluate(model, loader_blur, device)
    except Exception as e:
        print(f"Error: {e}")
        acc = 0 # Set to 0 if evaluation fails
    acc_vs_sigma.append(acc)
    print(f"sigma={s:.1f} -> acc {acc:.2f}%")

plt.figure(figsize=(6,4))
plt.plot(sigmas, acc_vs_sigma, marker="o")
plt.xlabel("Gaussian blur sigma"); plt.ylabel("Accuracy (%)")
plt.title("MNIST robustness to blur"); plt.grid(True, ls=":")
plt.show()


## Step 5 - Robustness Test the Model Against a Transform of Your Choice

Repeat step 4 for a transform of your choice. Create a plot showing how model accuracy degrades as the transform is applied more drastically.


In [ ]:
# Your code here

contrasts = [0.2, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
acc_vs_contrast = []

for c in contrasts:
    extra = ColorJitter(contrast=c)
    loader_c = make_test_loader_with(extra)
    loss, acc = evaluate(model, loader_c, device)
    acc_vs_contrast.append(acc)
    print(f"contrast={c:.1f} -> acc {acc:.2f}%")

plt.figure(figsize=(6,4))
plt.plot(contrasts, acc_vs_contrast, marker="o")
plt.xlabel("Contrast factor"); plt.ylabel("Accuracy (%)")
plt.title("MNIST robustness to contrast"); plt.grid(True, ls=":")
plt.show()
